# G1 Academy Bonus - Task 12: usage of the perception and pose-estimation pipeline

## Introduction
`util.DeliveryPipeline` is the one piece of academy infrastructure this track does not ask you to rebuild - OpenAI vision object detection, ArUco marker pose estimation, camera-to-base and wrist-to-palm transforms - because it is supplied, calibrated infrastructure, not a DDS/SDK concept. This task is about *using* it correctly: constructing it with the academy calibration values, running detection and pose estimation, validating confidence/freshness before acting, and understanding where it hands off to the IK helpers from Task 10. The full pick-and-place sequence built from these pieces is Task 13.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Construct the pipeline with calibration + get a frame
`camera_matrix`/`distortion` are the camera intrinsics; `camera_to_base`/`wrist_to_palm` are 4x4 transforms from the academy calibration file. Never leave placeholder identity/zero transforms in place for a real grasp - they are shown here only so the cells are syntactically complete.

In [ ]:
import numpy as np
import cv2
from util import DeliveryPipeline

camera_matrix = np.array([[605.0, 0.0, 320.0], [0.0, 605.0, 240.0], [0.0, 0.0, 1.0]])  # replace with the calibration file values
distortion = np.zeros(5)  # replace with the calibration file values
camera_to_base = np.eye(4)  # replace with the calibrated camera_to_base transform
wrist_to_palm = np.eye(4)  # replace with the calibrated wrist_to_palm transform

pipeline = DeliveryPipeline(camera_matrix, distortion, camera_to_base, wrist_to_palm)

import struct
def get_rgbd(endpoints=("tcp://127.0.0.1:5555", "tcp://localhost:5555")):
    import zmq
    ctx = zmq.Context.instance()
    for endpoint in endpoints:
        sock = ctx.socket(zmq.SUB)
        sock.setsockopt(zmq.SUBSCRIBE, b"")
        sock.setsockopt(zmq.RCVTIMEO, 3000)
        try:
            sock.connect(endpoint)
            parts = sock.recv_multipart()
            scale = struct.unpack("f", parts[2])[0] if parts[2] != b"0" and len(parts[2]) == 4 else None
            return {"timestamp": time.time(), "rgb_jpeg": bytes(parts[0]), "depth_png": None if parts[1] == b"0" else bytes(parts[1]), "depth_scale": scale}
        except Exception:
            continue
        finally:
            sock.close(0)
    return None

# frame = get_rgbd()
# detection = pipeline.detect_object(frame["rgb_jpeg"], "the small delivery package on the table")
# print(detection.label, detection.confidence, detection.center_px)

## Task 2 - ArUco pose estimation + transform to the base/palm frame
`aruco_pose` returns the marker pose in the *camera* frame; `marker_to_palm_target` chains `camera_to_base` then `wrist_to_palm` to produce the target in the frame the IK helper from Task 10 expects.

In [ ]:
# marker_length_m = 0.04  # measured marker edge length, meters
# rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
# marker_pose_camera = pipeline.aruco_pose(rgb, marker_length_m)
# palm_target = pipeline.marker_to_palm_target(marker_pose_camera)
# print("target in base_palm frame:", palm_target.translation_m)

## Task 3 - Validate confidence and freshness before acting
Perception output must never drive motion unchecked: reject a stale frame and reject a low-confidence detection instead of quietly acting on a guess.

In [ ]:
def validated_target(frame, marker_length_m, min_confidence=0.5, max_age_s=1.0, prompt="the delivery package"):
    if time.time() - frame.get("timestamp", time.time()) > max_age_s:
        raise RuntimeError("RGB-D frame is stale.")
    rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
    detection = pipeline.detect_object(frame["rgb_jpeg"], prompt)
    if detection.confidence < min_confidence:
        raise RuntimeError(f"Low-confidence detection ({detection.confidence}); refusing to act.")
    marker_pose = pipeline.aruco_pose(rgb, marker_length_m)
    return pipeline.marker_to_palm_target(marker_pose)

# target = validated_target(get_rgbd(), marker_length_m=0.04)

## Where this hands off
`pipeline.execute_incremental_ik(ik_increment, current_palm_xyz, target, side=...)` walks a bounded direct IK increment function (built from `ik_move_ee` in Task 10, or `util.make_recognition_ik_increment` for the `recognition_app_v3`-style executor) toward `target` in `max_step_m`-sized steps. Task 13 wires this together with SLAM navigation and Dex3 gripping into a full delivery sequence.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.